# Dedalus Chebyshev–Gauss to Chebyshev–Lobatto preprocessing

This test path reads the final Dedalus snapshot directly from scratch, retains the nonuniform Chebyshev representation, and spectrally resamples only the vertical axis from Dedalus's Chebyshev–Gauss points to the Chebyshev–Lobatto points expected by the quadrature-enabled PySPIDER fork. It does **not** create a uniform vertical grid.

In [ ]:
import gc
import h5py
import numpy as np
from numpy.polynomial.chebyshev import chebpts1, chebpts2
from scipy.fft import dct

source_path = "/storage/scratch1/2/ashelestov3/snaps_paper/snaps_paper_s1.h5"
output_path = "/storage/scratch1/2/ashelestov3/preprocessed_paper_data_lobatto.npz"
z_interval = (0.0, 1.0)

In [ ]:
print(f"Reading {source_path}")
with h5py.File(source_path, "r") as snapshots:
    tasks = snapshots["tasks"]
    scales = snapshots["scales"]
    print("Tasks:", list(tasks.keys()))

    velocity = np.asarray(tasks["velocity"])
    pressure = np.asarray(tasks["pressure"])
    buoyancy = np.asarray(tasks["buoyancy"])
    vorticity = np.asarray(tasks["vorticity"])

    x = np.asarray(scales["x_hash_4846e26f3f50e48ca5da4b8e0970d850261c704a"])
    y_gauss = np.asarray(scales["z_hash_050b2c5be05fb65046d37e4f71ee7ab1962cccdc"])
    t = np.asarray(scales["sim_time"])

# Match the archive convention used by the existing preprocessing notebooks.
# The convection notebook reverses this historical U/V name swap when loading.
horizontal_velocity = np.transpose(velocity[:, 0, :, :], (1, 2, 0))
vertical_velocity = np.transpose(velocity[:, 1, :, :], (1, 2, 0))
U = vertical_velocity
V = horizontal_velocity
P = np.transpose(pressure, (1, 2, 0))
B = np.transpose(buoyancy, (1, 2, 0))
W = np.transpose(vorticity, (1, 2, 0))
del velocity, pressure, buoyancy, vorticity, horizontal_velocity, vertical_velocity
gc.collect()

assert U.shape == V.shape == P.shape == B.shape == W.shape
assert U.shape == (x.size, y_gauss.size, t.size)
assert np.all(np.diff(x) > 0)
assert np.all(np.diff(y_gauss) > 0)
assert np.all(np.diff(t) > 0)

nz = y_gauss.size
a, b = z_interval
y_gauss_expected = 0.5 * (a + b) + 0.5 * (b - a) * chebpts1(nz)
if not np.allclose(y_gauss, y_gauss_expected, rtol=0.0, atol=5e-12):
    raise ValueError("Dedalus z coordinates do not match the expected Chebyshev–Gauss grid")

y_lobatto = 0.5 * (a + b) + 0.5 * (b - a) * chebpts2(nz)
print(f"Loaded arrays with shape {U.shape}")
print(f"Gauss z range:   [{y_gauss[0]:.16g}, {y_gauss[-1]:.16g}]")
print(f"Lobatto z range: [{y_lobatto[0]:.16g}, {y_lobatto[-1]:.16g}]")

In [ ]:
def chebyshev_gauss_to_lobatto(values, axis=1):
    """Spectrally evaluate Gauss-grid data on an equal-sized Lobatto grid."""
    values = np.asarray(values)
    n = values.shape[axis]
    if n < 2:
        raise ValueError("At least two vertical points are required")

    # scipy's DCT-II convention follows descending Gauss points. Work on a
    # separate buffer so the source array can be released after assignment.
    work = np.flip(values, axis=axis).copy()
    coeff = dct(work, type=2, axis=axis, overwrite_x=True) / n
    del work

    index = [slice(None)] * coeff.ndim
    index[axis] = 0
    coeff[tuple(index)] *= 0.5

    # A DCT-I evaluates the Chebyshev series at descending Lobatto points
    # when its interior coefficient inputs are halved.
    if n > 2:
        index[axis] = slice(1, -1)
        coeff[tuple(index)] *= 0.5
    lobatto_descending = dct(coeff, type=1, axis=axis, overwrite_x=True)
    return np.flip(lobatto_descending, axis=axis)

# Cheap analytic checks of the transform before applying it to the snapshots.
probe_gauss = y_gauss[None, :, None]
probe_lobatto = chebyshev_gauss_to_lobatto(probe_gauss)
np.testing.assert_allclose(probe_lobatto[0, :, 0], y_lobatto, rtol=2e-12, atol=2e-12)

for name in ("U", "V", "P", "B", "W"):
    print(f"Resampling {name}: Gauss -> Lobatto")
    globals()[name] = chebyshev_gauss_to_lobatto(globals()[name], axis=1)
    gc.collect()

y = y_lobatto
del y_lobatto
print("Spectral resampling complete:", U.shape)

In [ ]:
print(f"Saving Lobatto-grid arrays to {output_path}")
np.savez_compressed(
    output_path,
    U=U,
    V=V,
    P=P,
    B=B,
    W=W,
    x=x,
    y=y,
    t=t,
    z_grid_kind="chebyshev_lobatto",
    z_interval=np.asarray(z_interval),
    source_path=source_path,
)
print("Saved", output_path)